In [1]:
import geopandas as gpd
import folium
from geopy.geocoders import Nominatim
import xml.etree.ElementTree as ET
import sqlite3

In [2]:
def read_places(who="tim"):
    """
    return two lists. First is countries, the 2nd is cities.
    """
    tree = ET.parse("visited.xml")
    root = tree.getroot()
    rv = []
    cities = []
    username = who
    print(f"Filtering for {username}")
    for user in root.findall("user"):
        if user.get("username") != username:
            continue
        print(f"Places visited by {username}:")

        places = user.find("visited")
        for place in places.findall("place"):
            country = place.get("country")
            rv.append(country)
            for city in place.findall("city"):
                name = city.get("name")
                date_element = city.find("date")
                date = date_element.text if date_element is not None else None

                cities.append({
                    "city": name,
                    "country": country,
                    "date": date
                })
            date_element = place.find("date")
            if date_element is not None:
                date = date_element.text
            else:
                date = None

            print(country, date)
            rv.append(country)
  
    return list(set(rv)),cities


In [3]:

# Load world boundaries
world = gpd.read_file("ne_110m_admin_0_countries.zip")
# Get the data for user
visited,cities = read_places(who="tim")

# Separate visited countries using the A2 code i.e. FR, DE or NL 
visited_gdf = world[world["ISO_A2"].isin(visited)]


Filtering for tim
Places visited by tim:
FR None
US None
SA None
BH None
QA 1992
KW 1992
ES 1992
GB None
NL None
CZ None
DE None
AT None
CH None
IT None
VA None
BE None
EG None
BR None
AR None
GI None
MA None
KE None
TZ None
ZA None
OM None
AE None
IN None
LK None
MM None
NP None
TH None
SG None
KH None
VN None
KR None
JP None
HK None
MO None
TW None
PH None
SC None


In [4]:
def city_position(cities) -> list:
    """
    read_places returns a cities value (2nd value)
    Using this, we connect to a local SQLITE Db, and look up the city name.
    If it is in the local db, we add the city along with the lat and the long
    to an output list of dictionary objects. 

    Output will look something like this
    [{'city': 'Bilbao', 'lat': 43.26271, 'lon': -2.92528},
     {'city': 'London', 'lat': 51.50853, 'lon': -0.12574}]
    
    """
    conn = sqlite3.connect('cities.db')
    cursor = conn.cursor()
    query="SELECT * FROM cities where name=? and country_code=?"
    city_plot=[]
    for city in cities:    
        cursor.execute(query, (city['city'], city['country']))
        results = cursor.fetchall()
        if len(results)==1:
            print(f"{results}")
            city_plot.append({"city":city['city'],"lat":results[0][6],"lon":results[0][7]})
        else:
            print(f"Unk: {city['city']} {city['country']}")
    return city_plot

city_plot = city_position(cities)

[(2983990, 'Rennes', 'Rennes', 'FR', '53', '35', 48.11109, -1.67431, 227830, 'Europe/Paris')]
[(2990969, 'Nantes', 'Nantes', 'FR', '52', '44', 47.21725, -1.55336, 325070, 'Europe/Paris')]
[(3031582, 'Bordeaux', 'Bordeaux', 'FR', '75', '33', 44.84124, -0.58046, 265328, 'Europe/Paris')]
[(2996944, 'Lyon', 'Lyon', 'FR', '84', '69', 45.74906, 4.84789, 520774, 'Europe/Paris')]
[(2990440, 'Nice', 'Nice', 'FR', '93', '06', 43.70313, 7.26608, 342669, 'Europe/Paris')]
Unk: Chernourg FR
Unk: Boston US
Unk: Miami US
Unk: Houston US
[(4335045, 'New Orleans', 'New Orleans', 'US', 'LA', '071', 29.95465, -90.07507, 362701, 'America/Chicago')]
Unk: Salem US
Unk: New York US
Unk: Washington US
Unk: Las Vegas US
Unk: Denver US
[(5417598, 'Colorado Springs', 'Colorado Springs', 'US', 'CO', '041', 38.83388, -104.82136, 456568, 'America/Denver')]
[(108410, 'Riyadh', 'Riyadh', 'SA', '10', '', 24.68773, 46.72185, 4205961, 'Asia/Riyadh')]
Unk: Al Jubayl SA
[(106297, 'Hafar Al-Batin', 'Hafar Al-Batin', 'SA', '

In [5]:

# Create map
m = folium.Map(location=[30, 0], zoom_start=2, tiles="CartoDB positron")

# Add all countries in grey
folium.GeoJson(
    world,
    style_function=lambda feature: {
        "fillColor": "#dddddd",
        "color": "#888888",
        "weight": 0.5,
        "fillOpacity": 0.5,
    },
).add_to(m)

In [6]:

# Highlight visited countries
folium.GeoJson(
    visited_gdf,
    style_function=lambda feature: {
        "fillColor": "#3388ff",
        "color": "#0055aa",
        "weight": 2,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(fields=["NAME"], aliases=["Visited:"]),
).add_to(m)
## --------------------------------------------------
# Add city markers
# --------------------------------------------------


In [7]:
for city in city_plot:
        popup = f"<b>{city['city']}</b><br>"
    
        folium.CircleMarker(
            location=[
                city["lat"],
                city["lon"]
            ],
            radius=5,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.9,
            popup=popup
        ).add_to(m)


In [8]:
m.save("tim_visited.html")

In [9]:
print("All Done")

All Done
